In [20]:
import boto3
import sagemaker

from sagemaker.session import Session
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, TransformStep
from sagemaker.workflow.parameters import ParameterString, ParameterFloat
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.functions import JsonGet

from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.estimator import Estimator
from sagemaker.model import Model
from sagemaker.transformer import Transformer
from sagemaker.inputs import TrainingInput
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.functions import JsonGet, Join
from sagemaker.inputs import TransformInput

region = boto3.Session().region_name
role = sagemaker.get_execution_role()

sagemaker_session = Session()
pipeline_session = PipelineSession()

bucket = sagemaker_session.default_bucket()

print("Region:", region)
print("Role:", role)
print("Bucket:", bucket)

Region: us-east-1
Role: arn:aws:iam::864475311845:role/SageMakerStudioExecutionRole2026
Bucket: sagemaker-us-east-1-864475311845


In [21]:
input_data = ParameterString(
    name="InputData",
    default_value=f"s3://{bucket}/future-sales/raw/"
)

rmse_threshold = ParameterFloat(
    name="RMSEThreshold",
    default_value=1.0
)

In [22]:
preprocess_image = "864475311845.dkr.ecr.us-east-1.amazonaws.com/future-sales-processing:latest"
train_image = "864475311845.dkr.ecr.us-east-1.amazonaws.com/future-sales-byoc:latest"
inference_image = "864475311845.dkr.ecr.us-east-1.amazonaws.com/future-sales-byoc:latest"

In [23]:
processor = ScriptProcessor(
    image_uri=preprocess_image,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    sagemaker_session=pipeline_session,
)

step_preprocess = ProcessingStep(
    name="PreprocessStep",
    processor=processor,
    inputs=[
        ProcessingInput(
            source=input_data,
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train_data",
            source="/opt/ml/processing/output/train",
        ),
        ProcessingOutput(
            output_name="validation_data",
            source="/opt/ml/processing/output/validation",
        ),
        ProcessingOutput(
            output_name="test_data",
            source="/opt/ml/processing/output/test",
        ),
        
        ProcessingOutput(
    output_name="batch_data",
    source="/opt/ml/processing/output/batch",
),
    ],
    code="../processing/src/preprocess.py",
)

In [24]:
estimator = Estimator(
    image_uri=train_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/future-sales/model-artifacts/",
    sagemaker_session=pipeline_session,
)

step_train = TrainingStep(
    name="TrainStep",
    estimator=estimator,
    inputs={
        "train": TrainingInput(
            s3_data=step_preprocess.properties.ProcessingOutputConfig.Outputs[
                "train_data"
            ].S3Output.S3Uri
        ),
        "validation": TrainingInput(
            s3_data=step_preprocess.properties.ProcessingOutputConfig.Outputs[
                "validation_data"
            ].S3Output.S3Uri
        ),
    },
)

In [25]:
evaluation_processor = ScriptProcessor(
    image_uri=train_image,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    sagemaker_session=pipeline_session,
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json"
)

step_evaluate = ProcessingStep(
    name="EvaluateStep",
    processor=evaluation_processor,
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/input/model",
        ),
        ProcessingInput(
            source=step_preprocess.properties.ProcessingOutputConfig.Outputs[
                "test_data"
            ].S3Output.S3Uri,
            destination="/opt/ml/processing/input/test",
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/output/evaluation",
        ),
    ],
    code="../src/evaluation/evaluate.py",
    property_files=[evaluation_report],
)

In [26]:
model = Model(
    image_uri=inference_image,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    sagemaker_session=pipeline_session,
)

step_model = ModelStep(
    name="CreateModelStep",
    step_args=model.create(instance_type="ml.m5.large"),
)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [47]:
transformer = Transformer(
    model_name=step_model.properties.ModelName,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/future-sales/batch-output/",
    strategy="SingleRecord",
    sagemaker_session=pipeline_session,
)

step_transform = TransformStep(
    name="BatchTransformStep",
    step_args=transformer.transform(
        data=step_preprocess.properties.ProcessingOutputConfig.Outputs[
            "batch_data"
        ].S3Output.S3Uri,
        content_type="application/json",
        split_type="Line",
    ),
)

In [48]:
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(
            on="/",
            values=[
                step_evaluate.properties.ProcessingOutputConfig.Outputs[
                    "evaluation"
                ].S3Output.S3Uri,
                "evaluation.json",
            ],
        ),
        content_type="application/json",
    )
)

step_register = ModelStep(
    name="RegisterModelStep",
    step_args=model.register(
        content_types=["text/csv"],
        response_types=["text/csv"],
        inference_instances=["ml.m5.large"],
        transform_instances=["ml.m5.large"],
        model_package_group_name="future-sales-model-package-group",
        approval_status="PendingManualApproval",
        model_metrics=model_metrics,
    ),
)

In [49]:
step_fail = FailStep(
    name="FailStep",
    error_message="El RMSE del modelo es mayor al threshold definido."
)

In [50]:
step_condition = ConditionStep(
    name="CheckRMSE",
    conditions=[
        ConditionLessThanOrEqualTo(
            left=JsonGet(
                step_name=step_evaluate.name,
                property_file=evaluation_report,
                json_path="regression_metrics.rmse.value",
            ),
            right=rmse_threshold,
        )
    ],
    if_steps=[step_transform, step_register],
    else_steps=[step_fail],
)

In [51]:
pipeline = Pipeline(
    name="future-sales-byoc-pipeline",
    parameters=[input_data, rmse_threshold],
    steps=[
        step_preprocess,
        step_train,
        step_evaluate,
        step_model,
        step_condition,
    ],
    sagemaker_session=pipeline_session,
)

In [52]:
pipeline.upsert(role_arn=role)

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:864475311845:pipeline/future-sales-byoc-pipeline',
 'ResponseMetadata': {'RequestId': 'c31b7732-c404-4dca-ab3b-d03456b061a5',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'c31b7732-c404-4dca-ab3b-d03456b061a5',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '117',
   'date': 'Wed, 25 Mar 2026 18:22:49 GMT'},
  'RetryAttempts': 0}}

In [53]:
execution = pipeline.start()
print(execution.arn)

arn:aws:sagemaker:us-east-1:864475311845:pipeline/future-sales-byoc-pipeline/execution/xsmw34l7bznn


In [54]:
for step in execution.list_steps():
    print(f"{step['StepName']} → {step['StepStatus']}")

PreprocessStep → Starting


In [55]:
import time

while True:
    status = execution.describe()["PipelineExecutionStatus"]
    print(f"Pipeline status: {status}")

    for step in execution.list_steps():
        print(f"{step['StepName']} → {step['StepStatus']}")
    print("-" * 80)

    if status in ["Succeeded", "Failed", "Stopped"]:
        break

    time.sleep(20)

Pipeline status: Executing
PreprocessStep → Executing
--------------------------------------------------------------------------------
Pipeline status: Executing
PreprocessStep → Executing
--------------------------------------------------------------------------------
Pipeline status: Executing
PreprocessStep → Executing
--------------------------------------------------------------------------------
Pipeline status: Executing
PreprocessStep → Executing
--------------------------------------------------------------------------------
Pipeline status: Executing
PreprocessStep → Executing
--------------------------------------------------------------------------------
Pipeline status: Executing
PreprocessStep → Executing
--------------------------------------------------------------------------------
Pipeline status: Executing
PreprocessStep → Executing
--------------------------------------------------------------------------------
Pipeline status: Executing
PreprocessStep → Executing
-

In [57]:
for step in execution.list_steps():
    if step["StepStatus"] == "Failed":
        print("STEP:", step["StepName"])
        print("FAILURE:", step.get("FailureReason", ""))